# Phases 3–6: dry-run, probe, bulk load, verification

Follow-along driver for `update_mailing_addresses.py` — the script stays the single
source of truth; this notebook imports its functions and walks the run step by step.

Sections 1–3 are local-only. Section 4 authenticates and runs **read-only** SOQL.
Sections 5 (probe: ONE record) and 6 (bulk load, ~147 jobs) **write to production**
and are each gated behind an explicit flag you have to flip by hand.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # repo root
sys.path.insert(0, str(Path.cwd()))         # this folder: update_mailing_addresses

import pandas as pd

import update_mailing_addresses as um
from config import load_mysql_config
from mysql_client import MySQLClient

BATCH_ID = "2026-08-25_billing_backfill"

# Phase 2 contract: fill in the staged row count from 03 after staging is frozen.
# None = staging not frozen yet, the assert below is skipped.
EXPECTED_ROWS = None

db = MySQLClient(load_mysql_config())
print("connected | batch:", BATCH_ID)

## 1. Load the staged batch (local, read-only)

Same row selection the script uses: update rows, not excluded, street present, not yet
processed. After a successful load the open count drops to 0 — flip `AFTER_LOAD = True`
for post-load reruns.

In [ ]:
AFTER_LOAD = False  # True once the bulk load ran: open rows are then expected to be 0

rows = db.fetch_all("""
    SELECT * FROM crm_imp_person_accounts
    WHERE _batch_id = %s AND _operation = 'update' AND _excluded = 0
      AND _billing_processed_at IS NULL
      AND sf_account_id IS NOT NULL
      AND address IS NOT NULL AND address <> ''
""", (BATCH_ID,))
print(f"{len(rows):,} rows open")

if not AFTER_LOAD and EXPECTED_ROWS is not None:
    assert len(rows) == EXPECTED_ROWS, "open rows differ from the phase 2 contract"
    print("matches the phase 2 contract")

## 2. Build the payload and eyeball it (local)

`row_to_sf_record` is the exact mapper the load uses: only `Id` plus the four
`PersonMailing*` fields, and a field is **omitted entirely** when the source is empty
(a Bulk-API-update empty cell would CLEAR the target field). The signature breakdown
shows how many Bulk jobs per field combination the load will submit — no CSV ever
contains an empty cell.

In [ ]:
records = [um.row_to_sf_record(r) for r in rows]
groups = um.group_by_signature(records)
print("signature groups (fields -> records):")
for sig, recs in sorted(groups.items(), key=lambda kv: -len(kv[1])):
    print(f"  {list(sig)}: {len(recs):,}")

preview = pd.DataFrame(records[:5])
preview

## 3. Phase 3 — dry-run (local, no Salesforce contact)

Runs the script itself with `--dry-run`: builds one CSV per signature group under
`<repo-root>/local_data/dry_run_billing_backfill_*.csv`. Open the biggest one and check:
header = `Id,PersonMailingCity,PersonMailingCountry,PersonMailingPostalCode,PersonMailingStreet`,
no empty cells anywhere, ISO-2 codes in the country column.

In [ ]:
import subprocess

proc = subprocess.run(
    [sys.executable, str(Path.cwd() / "update_mailing_addresses.py"), BATCH_ID, "--dry-run"],
    cwd=str(Path.cwd().parent), capture_output=True, text=True,
)
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr)
    raise RuntimeError(f"dry-run exited {proc.returncode}")

## 4. Authenticate + live spot-check (prod, READ-ONLY)

Samples 500 staged accounts and runs the same live check the loader runs on everything:
how many already have a `PersonMailingStreet` in prod (written by the live integration
since the mirror refresh)? A handful is normal drift; a large share means the mirror is
stale — re-refresh and re-stage.

In [ ]:
from salesforce_client_prod import SalesforceClientCC, load_salesforce_cc_config_from_env

sf = SalesforceClientCC(load_salesforce_cc_config_from_env())
sf.authenticate()
print("authenticated")

sample_ids = [str(r["sf_account_id"]) for r in rows[:500]]
taken = um.accounts_with_mailing_street(sf, sample_ids)
print(f"live check on {len(sample_ids)} sampled accounts: {len(taken)} already have PersonMailingStreet")

## 5. Phase 4 — probe record (prod, WRITES ONE ACCOUNT)

Put the agreed test account's 18-char Id into `PROBE_ACCOUNT_ID` (it must be in the
open batch), then flip `RUN_PROBE = True`. Uses a single REST PATCH — same field
payload as the bulk load. The readback afterwards shows all four PersonMailing fields
AND the Billing side, to prove Billing stayed untouched. Eyeball it in the UI too,
then business sign-off **before** section 6.

In [ ]:
RUN_PROBE = False          # <- flip by hand for the one probe update
PROBE_ACCOUNT_ID = ""      # <- the agreed test account's 18-char Id, from the open batch

if RUN_PROBE:
    candidates = [r for r in rows if str(r["sf_account_id"]) == PROBE_ACCOUNT_ID]
    assert candidates, f"account {PROBE_ACCOUNT_ID!r} is not in the open batch"
    probe_row = candidates[0]
    payload = {k: v for k, v in um.row_to_sf_record(probe_row).items() if k != "Id"}
    print("payload:", payload)
    r = sf._client.patch(f"{sf._base()}/sobjects/Account/{PROBE_ACCOUNT_ID}", json=payload)
    r.raise_for_status()
    print("probe account updated:", PROBE_ACCOUNT_ID, "| status:", r.status_code)
else:
    print("probe skipped (RUN_PROBE = False)")

In [ ]:
if RUN_PROBE:
    rec = sf.query_all(
        f"SELECT Id, PersonEmail, PersonMailingStreet, PersonMailingCity, "
        f"PersonMailingPostalCode, PersonMailingCountry, "
        f"BillingStreet, BillingCity, BillingPostalCode, BillingCountryCode__c, "
        f"LastModifiedDate FROM Account WHERE Id = '{PROBE_ACCOUNT_ID}'"
    )["records"]
    for x in rec:
        x.pop("attributes", None)
    print("— account as Salesforce stored it —")
    display(pd.DataFrame(rec).T)
    print("check: all four PersonMailing fields filled from the Billing values,")
    print("       Billing side unchanged, country is the ISO-2 code")

## 6. Phase 5 — bulk load (prod, WRITES ~732k ACCOUNTS)

Runs the script itself, so the real run is exactly what was dry-run — plus the full
live skip-check (accounts whose PersonMailingStreet got filled since the mirror are
skipped and logged), the duplicate-Id abort, per-signature jobs of 5,000, and the
per-job `_billing_processed_at` writeback that makes a crashed run resumable by
re-running the same command. The script exits non-zero on ANY record failure or job
error. Expect ~147 jobs, roughly 1–3 hours; monitor in Setup -> Bulk Data Load Jobs.

**Does not run without Arsal's explicit go-ahead.** Flip `RUN_LOAD = True` only then.
Note: the probe account is already updated, so the live skip-check will skip it — that
one skip is expected.

In [ ]:
RUN_LOAD = False  # <- Arsal's explicit go-ahead required (phase 5 gate)

if RUN_LOAD:
    proc = subprocess.run(
        [sys.executable, str(Path.cwd() / "update_mailing_addresses.py"), BATCH_ID],
        cwd=str(Path.cwd().parent), capture_output=True, text=True,
    )
    print(proc.stdout[-8000:])
    if proc.returncode != 0:
        print(proc.stderr)
        raise RuntimeError(f"load exited {proc.returncode} - fix before re-running")
else:
    print("bulk load skipped (RUN_LOAD = False)")

## 7. Phase 6 — verification (prod, read-only)

Live SOQL: how many accounts still match the population rule (BillingStreet set,
PersonMailingStreet empty)? Expect roughly the live-skipped count plus whatever the
integration created since the load. Plus staging writeback completeness
(`still_open` must be 0).

In [ ]:
remaining = sf.query_all(
    "SELECT COUNT(Id) n FROM Account "
    "WHERE BillingStreet != null AND PersonMailingStreet = null"
)["records"][0]["n"]
print(f"live accounts still billing-only: {remaining:,}")

open_rows = db.fetch_df("""
    SELECT SUM(_billing_processed_at IS NULL) AS still_open, COUNT(*) AS total
    FROM crm_imp_person_accounts
    WHERE _batch_id = %s
""", (BATCH_ID,))
print(open_rows.to_string(index=False))
print("\nmismatch? check <repo-root>/local_data/skipped_* and failed_* before touching anything")